# Crypto RL Trading Bot — PPO Training on Colab

**Before running:** `Runtime → Change runtime type → T4 GPU`

All 3 models train in parallel and are auto-saved to `MyDrive/cryptobot-rl/models/` so they survive session disconnects.

In [ ]:
# 0. Mount Google Drive — models saved here survive session disconnects
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_MODELS_DIR = '/content/drive/MyDrive/cryptobot-rl/models'
os.makedirs(DRIVE_MODELS_DIR, exist_ok=True)
print(f'Drive mounted. Models will be saved to: {DRIVE_MODELS_DIR}')

In [ ]:
# 1. Verify T4 GPU, clone repo, install dependencies
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('WARNING: No GPU detected — go to Runtime → Change runtime type → T4 GPU')

REPO_URL = 'https://github.com/HussianRaza/cryptobot-rl.git'
!git clone {REPO_URL} cryptobot-rl
%cd cryptobot-rl
!pip install -q -r requirements.txt

In [ ]:
# 2. Download OHLCV data and compute indicators (per-split, no lookahead)
!python scripts/download_data.py --assets btc eth sol
!python scripts/compute_indicators.py

import pandas as pd
df = pd.read_csv('data/processed/btc_features.csv')
print('BTC features shape:', df.shape, '| NaNs:', df.isna().sum().sum())

In [ ]:
# 3. Smoke-test the Gymnasium environment (check_env + 100 random steps)
!python scripts/smoke_test_env.py btc

In [ ]:
# 4. Train BTC + ETH + SOL in PARALLEL — all 3 share the T4 GPU simultaneously
#
# Config per asset:
#   --n-envs 2       2 envs per process (6 total across 3 processes)
#   --n-steps 4096   larger rollout buffer -> bigger GPU batches
#   --batch-size 512 saturates T4 (8192 steps / 512 = 16 minibatch updates/epoch)
#
# Each model is copied to Google Drive immediately on completion.

import subprocess, time, shutil, os

os.makedirs('logs', exist_ok=True)
os.makedirs('models', exist_ok=True)

ASSETS = ['btc', 'eth', 'sol']
BASE_CMD = [
    'python', 'scripts/train_ppo.py',
    '--timesteps', '500000',
    '--seed', '42',
    '--tag', 'final',
    '--n-envs', '2',
    '--n-steps', '4096',
    '--batch-size', '512',
]

# Launch all 3 simultaneously
procs = {}
for asset in ASSETS:
    log_fh = open(f'logs/train_{asset}.log', 'w')
    p = subprocess.Popen(BASE_CMD + ['--asset', asset], stdout=log_fh, stderr=subprocess.STDOUT)
    procs[asset] = {'proc': p, 'log_fh': log_fh}
    print(f'  [{asset.upper()}] started (PID {p.pid})')

print('\nPolling every 60s — T4 GPU shared across all 3 jobs...\n')

done_set = set()
while len(done_set) < len(ASSETS):
    time.sleep(60)
    for asset, info in procs.items():
        if asset in done_set:
            continue
        rc = info['proc'].poll()
        with open(f'logs/train_{asset}.log') as f:
            lines = f.readlines()
        tail = ' | '.join(l.strip() for l in lines[-2:] if l.strip())
        if rc is None:
            print(f'  [{asset.upper()}] running ... {tail}')
        else:
            info['log_fh'].close()
            model_path = f'models/ppo_{asset}_final.zip'
            if os.path.exists(model_path):
                shutil.copy(model_path, DRIVE_MODELS_DIR)
                kb = os.path.getsize(model_path) / 1024
                print(f'  [{asset.upper()}] DONE ({kb:.0f} KB) -> copied to Drive')
            else:
                print(f'  [{asset.upper()}] FAILED (rc={rc}) — see logs/train_{asset}.log')
            done_set.add(asset)
    print()

print('All training complete!')

In [ ]:
# 5. Verify all 3 models exist (local + Drive)
import os
for asset in ['btc', 'eth', 'sol']:
    local = f'models/ppo_{asset}_final.zip'
    drive_p = f'{DRIVE_MODELS_DIR}/ppo_{asset}_final.zip'
    local_ok = os.path.exists(local)
    drive_ok = os.path.exists(drive_p)
    src = drive_p if drive_ok else (local if local_ok else None)
    size = os.path.getsize(src) / 1024 if src else 0
    status = 'local+Drive' if (local_ok and drive_ok) else ('Drive only' if drive_ok else ('local only' if local_ok else 'MISSING'))
    print(f'  {asset.upper()}: {status} ({size:.0f} KB)')

In [ ]:
# 6. Run baselines on 2024 test split
!python scripts/run_baselines.py --asset btc --year 2024

In [ ]:
# 7. Download all 3 model ZIPs to your laptop
# Then place them in projectimplementation/models/ and commit to git
from google.colab import files
import os

for asset in ['btc', 'eth', 'sol']:
    drive_path = f'{DRIVE_MODELS_DIR}/ppo_{asset}_final.zip'
    local_path = f'models/ppo_{asset}_final.zip'
    src = drive_path if os.path.exists(drive_path) else (local_path if os.path.exists(local_path) else None)
    if src:
        kb = os.path.getsize(src) / 1024
        print(f'Downloading ppo_{asset}_final.zip ({kb:.0f} KB)...')
        files.download(src)
    else:
        print(f'WARNING: ppo_{asset}_final.zip not found — did training complete?')